<a href="https://colab.research.google.com/github/xiaofanpaiooo-code/whisper-lora/blob/main/%E5%9F%BA%E4%BA%8E%E6%B7%B1%E5%BA%A6%E5%AD%A6%E4%B9%A0%E7%9A%84%E6%99%BA%E8%83%BD%E8%AF%AD%E9%9F%B3%E5%AD%97%E5%B9%95%E7%B3%BB%E7%BB%9F1.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Cell 1] 环境准备与模型加载

In [1]:
!pip install datasets transformers librosa soundfile

In [14]:
# 确保已安装必要的库：!pip install datasets transformers librosa soundfile
from transformers import WhisperProcessor
from datasets import load_dataset, interleave_datasets, Audio

# 加载特征提取器与分词器 (以 whisper-small 为例)
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="English",
    task="transcribe"
)

[Cell 2] 数据处理

In [15]:
from datasets import load_dataset, interleave_datasets, Audio

# 1. 开启流式加载，彻底规避 OOM
ds_65h = load_dataset("gongqingyu/bishe_whisper_dataset_65h", split="train", streaming=True)
ds_35h = load_dataset("gongqingyu/bishe_whisper_dataset_35h2", split="train", streaming=True)

# ================= 核心修复：合并前强制对齐特征 Schema =================
# 步骤 A：强行将两个流的音频特征空间锚定为 16000Hz（Whisper的标准输入频率）
ds_65h = ds_65h.cast_column("audio", Audio(sampling_rate=16000))
ds_35h = ds_35h.cast_column("audio", Audio(sampling_rate=16000))

# 步骤 B：文本标签列名对齐 (⚠️ 强烈预警：Schema 必须100%一致)
# 假设你在构建数据时，ds_65h 的文本叫 "text"，而 ds_35h 叫 "transcription"
# 这里提供一个健壮的预处理，先检查列名并统一改名为 "transcription"
def align_text_column(dataset):
    column_names = list(dataset.features.keys())
    if "text" in column_names and "transcription" not in column_names:
        return dataset.rename_column("text", "transcription")
    elif "sentence" in column_names and "transcription" not in column_names:
        return dataset.rename_column("sentence", "transcription")
    return dataset

ds_65h = align_text_column(ds_65h)
ds_35h = align_text_column(ds_35h)

# ================= 移除无关的冗余列 (精简内存管线) =================
# 不同数据集可能带有特有的额外字段（如 client_id, up_votes 等），这些会导致 Schema 依然不匹配
# 我们只保留 ASR 所需的核心列：'audio' 和 'transcription'
columns_to_keep = ["audio", "transcription"]
ds_65h = ds_65h.select_columns(columns_to_keep)
ds_35h = ds_35h.select_columns(columns_to_keep)
# ====================================================================

# 2. 动态交替混合数据集 (此时 Schema 已处于绝对安全的强制一致状态)
mixed_dataset = interleave_datasets([ds_65h, ds_35h], probabilities=[0.65, 0.35], seed=42)

# 3. 局部缓冲区打乱
# 缓冲区设为1000，保障每个 mini-batch 存在领域数据与通用数据的黄金混合比
shuffled_dataset = mixed_dataset.shuffle(seed=42, buffer_size=1000)

print("✅ 流式数据集混合与预处理流图已成功重构并完成 Schema 对齐！")

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

✅ 流式数据集混合与预处理流图已成功重构并完成 Schema 对齐！


[Cell 3]定义特征映射管线

In [16]:
def prepare_dataset(batch):
    # 1. 批次解包：此时 batch["audio"] 是一个 list，里面包含多个音频字典
    # 使用列表推导式提取出当前 batch (16条) 所有的音频一维矩阵 array
    audio_arrays = [audio["array"] for audio in batch["audio"]]

    # 2. 批次特征提取：WhisperProcessor 天生支持传入 List[Array] 进行并行计算
    # 因为我们在 Cell 2 已经统一对齐为 16000Hz，这里可以直接写死 16000 节省提取开销
    extracted_features = processor.feature_extractor(
        audio_arrays,
        sampling_rate=16000
    )
    # 直接赋值整个批次的 input_features
    batch["input_features"] = extracted_features.input_features

    # 3. 批次文本分词：同样提取批次文本列表，并行 Tokenize
    text_strings = batch["transcription"]
    tokenized_labels = processor.tokenizer(text_strings)
    batch["labels"] = tokenized_labels.input_ids

    return batch

# 将向量化的映射函数应用到流式数据集中
# batched=True 保障了底层的 C++ 多线程吞吐，最大化规避 OOM 并缩短预处理时间
vectorized_ds = shuffled_dataset.map(prepare_dataset, batched=True, batch_size=16)

print("✅ 批次化特征映射函数已重新挂载！")

✅ 批次化特征映射函数已重新挂载！


[Cell 4] 特征维度验证

In [17]:
# 将 IterableDataset 转为迭代器，抓取第一个样本
sample_iterator = iter(vectorized_ds)
sample = next(sample_iterator)

# 验证核心特征
input_features = sample["input_features"]
labels = sample["labels"]

print("=== 特征映射维度验证报告 ===")
# 预期输出必须严格为: 80 x 3000
print(f"[声学特征] Log-Mel Spectrogram 维度: {len(input_features)} x {len(input_features[0])}")
print(f"[文本特征] Token IDs 长度: {len(labels)}")
print(f"[文本采样] 前 5 个 Token IDs: {labels[:5]}")
print("=============================")

=== 特征映射维度验证报告 ===
[声学特征] Log-Mel Spectrogram 维度: 80 x 3000
[文本特征] Token IDs 长度: 26
[文本采样] 前 5 个 Token IDs: [50258, 50259, 50359, 50363, 3322]


[Cell 5]数据收集器构建

In [18]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 1. 抽离声学特征与文本特征
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # 2. 将声学特征转换为 PyTorch Tensor (此时已经是 80x3000，无需额外 pad)
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 3. 动态填充文本标签至当前 Batch 的最大长度
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # 4. 将填充位的 Token ID（通常是 processor.tokenizer.pad_token_id）替换为 -100
        # 这是避免计算 Padding Loss 的核心数学操作
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # 5. Whisper 模型特有机制：如果序列起始标志位是 <|startoftranscript|>，将其裁掉，因为模型在 Decoder 端会自动添加
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

# 实例化数据收集器
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

[Cell 6]评估引擎配置

In [13]:
!pip install evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.7 MB/s eta 0:00:00


In [19]:
import evaluate
import numpy as np

# 1. 加载纯英文工业级标准评估指标 WER
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # 2. 掩码还原：将 -100 替换回 pad_token_id，避免解码器崩溃
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # 3. 批量解码 Token 为可读的英文字符串
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # 4. [统计学与工程优化点：文本归一化]
    # 在英文 ASR 中，模型可能会输出 "Python" 而标签是 "python"。
    # 为了防止大小写带来的“虚假错误”拉高 WER，我们通常在评估前强制小写化。
    pred_str = [s.lower().strip() for s in pred_str]
    label_str = [l.lower().strip() for l in label_str]

    # 5. 计算词错误率 (WER)
    wer = metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("✅ 纯英文 WER 评估引擎已成功挂载！")

✅ 纯英文 WER 评估引擎已成功挂载！
